# Compare
Compare the uncertainty ellipses to the NHC 5-year average circles.

In [ ]:
%matplotlib inline
%load_ext autotime
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import plots

In [ ]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

In [ ]:
FIGURE_PATH = "figures/analysis/"
PREDICTIONS_PATH = "saved_predictions/"

In [ ]:
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 600
np.warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

In [ ]:
ELLIPSE_COLOR = 'steelblue'
CIRCLE_COLOR = 'orange'
NAUTICAL_MILE_TO_KM = 1.852
FIGURE_WIDTH = 25

In [ ]:
R = np.sqrt(-2.0 * (np.log(1.0 - 0.67)))        # Pr(capture) = 67%

# The NHC circle radii are given in nautical miles.
NHC2017 = {
    ('AL',  3): 16, ('AL', 12):  29, ('AL', 24):  45, ('AL', 36):   63,
    ('AL', 48): 78, ('AL', 72): 107, ('AL', 96): 159, ('AL', 120): 211,
}

NHC2022 = {
    ('AL', 12): 26, ('AL', 24):  39, ('AL', 36):  52, ('AL', 48):   67,
    ('AL', 60): 84, ('AL', 72): 100, ('AL', 96): 142, ('AL', 120): 200,
}

In [ ]:
THETA = np.linspace(0, 2 * np.pi, 1000)

def plot_circle(ax, radius, label=None):
    x = radius * np.cos(THETA)
    y = radius * np.sin(THETA)
    ax.plot(x, y, '-', color=CIRCLE_COLOR, linewidth=4, label=label)


def plot_ellipse(ax, sigma_u, sigma_v, rho, r, label=None):
    x = r * sigma_u * np.cos(THETA)
    y = r * sigma_v * (rho * np.cos(THETA) + np.sqrt(1 - rho * rho) * np.sin(THETA))
    ax.plot(x, y, '-', color=ELLIPSE_COLOR, linewidth=1.5, label=label)


def equalize_subplot_limits(axes):
    """Set all the subplot axes limits equal."""
    all_left = all_right = all_top = all_bottom = 0.0

    for ax in axes:
        left, right = ax.get_xlim()
        bottom, top = ax.get_ylim()

        all_left   = min(left, all_left)
        all_right  = max(right, all_right)
        all_bottom = min(bottom, all_bottom)
        all_top    = max(top, all_top)

    for ax in axes:
        ax.set_xlim(all_left, all_right)
        ax.set_ylim(all_bottom, all_top)
        ax.grid(False)

In [ ]:
# Adjustable parameters
name = "IRMA"
basin = 'AL'
ftimes = [12, 24, 36, 48, 60, 72, 96, 120]
year = 2017  # 2022
nhc = NHC2017 # NHC2022
nhc_label = "NHC 2017 Cone"  # "NHC 2022 Cone"
rng_seed = 3

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(FIGURE_WIDTH, 0.5*FIGURE_WIDTH),
    facecolor="white",
)
ax = ax.flatten()

for i, ftime in enumerate(ftimes):
    pathname = f"{PREDICTIONS_PATH}/centered_bivariate_normal_*_AL{ftime}_{year}_centered_bivariate_normal_rng_seed_{rng_seed}_testing_predictions.csv"
    file_list = glob.glob(pathname=pathname)

    ax[i].set_aspect('equal')
    ax[i].set_title(f"{basin}{ftime} {name} {year}")

    # see if the file exists, if it does, load it.
    try:
        PREDICTIONS_FILE = file_list[0]
        predictions = pd.read_csv(PREDICTIONS_FILE)
    except:
        continue

    parameters = predictions.loc[predictions['Name']==name, ['time', 'sigma_u', 'sigma_v', 'rho']].copy()
    parameters.sort_values(by='time', inplace=True, ignore_index=True)

    for j, row in parameters.iterrows():
        plot_ellipse(
            ax[i],
            row['sigma_u'],
            row['sigma_v'],
            row['rho'],
            R,
            label="TCAN Uncertainty (67%)" if j==0 else None
        )

    plot_circle(ax[i], nhc[(basin, ftime)]*NAUTICAL_MILE_TO_KM, label=nhc_label)

equalize_subplot_limits(ax)
ax[0].legend()
plt.tight_layout()

exp_name = f"centered_bivariate_normal_{name}_AL_{year}_centered_bivariate_normal_rng_seed_{rng_seed}_testing_predictions"
plt.savefig(
    FIGURE_PATH + 'compare_uq_nhc_by_storm_' + exp_name + '.png',
    dpi=dpiFig,
    bbox_inches='tight',
)

plt.show()